## Obiettivi
Per il momento ci concentriamo sugli **algoritmi centralizzati di random sampling**, per poi ricollegarci al mondo distribuito.

Il problema di **random sampling** può essere formulato come segue: dato un dataset $A$ di grandi dimensioni ($|A| \gg 0$, $|A| = n$), vogliamo estrarre un campione $S$ di dimensione molto più piccola ($|S| = s$, con $s \ll n$) in modo che tale campione sia **rappresentativo** del dataset originale. 

Con **rappresentativo** intendiamo che il campione $S$ deve soddisfare precise e rigorose proprietà statistiche. Nel corso di Big Data è stato già affrontato l'algoritmo centralizzato di **Reservoir Sampling**, ma l'analisi è stata superficiale: ci eravamo limitati a studiare la probabilità marginale per cui $$\forall x \in A, \qquad \Pr(x \in S) = \frac{s}{n}$$
ossia per ogni elemento $x$ del dataset originale, la probabilità che $x$ sia incluso nel campione $S$ è esattamente $s/n$ (si parla di probabilità marginale perché stiamo guardando un solo elemento alla volta, ignorando le relazioni con gli altri elementi del sottinsieme $S$).

Il problema della probabilità marginale è che non basta a garantire che il campione $S$ sia davvero uniforme. Infatti questa controlla solo la probabilità che un singolo elemento sia incluso in $S$, ma non dice nulla sulle probabilità congiunte di più elementi che appaiono insieme in $S$. Potrebbe quindi accadere che ogni elemento abbia la giusta probabilità marginale di essere scelto, ma che poi alcuni sottinsiemi di cardinalità $s$ siano più probabili di altri, come si vede nell'esempio seguente:

es. ipotizziamo di avere un dataset $A = \{a, b, c, d\}$ e vogliamo estrarre un campione di dimensione $s = 2$. I Possibili sample in questo caso sono $\{a, b\}$, $\{a, c\}$, $\{a, d\}$, $\{b, c\}$, $\{b, d\}$ e $\{c, d\}$. Se ogni sample fosse uniforme allora ci aspetteremmo che ogni coppia appaia con probabilità $1/6$. Immaginiamo un algoritmo che sceglie solo le coppie $\{a, b\}$ e $\{c, d\}$ con probabilità $1/2$ ciascuna. In questo caso:
$$\Pr(a \in S) = \Pr(b \in S) = \Pr(c \in S) = \Pr(d \in S) = 1/2$$
quindi la probabilità marginale è corretta per ogni documento, però il sample non è uniforme perché quattro coppie hanno probabilità zero di essere estratte $\{a, c\}$, $\{a, d\}$, $\{b, c\}$ e $\{b, d\}$.

## Proprietà da soddisfare
**Per questo motivo vogliamo garantire la proprietà molto forte per cui il sample prodotto dall'algoritmo deve essere uniformemente distribuito su tutti i possibili sample di dimensione $s$**. 

Si scrive formalmente tale proprietà di uniformità come segue. Sia $\mathcal{S}$ la **variabile aleatoria** che rappresenta il sample prodotto dall'algoritmo. Vogliamo che $\mathcal{S}$ soddisfi la seguente proprietà di uniformità:
$$\forall S \subseteq A : |S| = s, \qquad \Pr(\mathcal{S} = S) = \frac{1}{\binom{n}{s}}$$
ossia ogni sottinsieme di cardinalità s deve avere esattamente la stessa probabilità di essere scelto (nota che $\mathcal{S}$ calligrafico è una variabile aleatoria che rappresenta il sample prodotto dall'algoritmo, mentre $S$ rappresenta un sample fisso, deterministico, di dimensione $s$).

Questa proprietà, essendo più forte della probabilità marginale, la implica.

Un'**ulteriore proprietà** statistica che vorremmo garantire riguarda **l'ordinamento degli elementi nel sample**. Finora abbiamo considerato il sample $S$ come un semplice sottinsieme non ordinato del dataset originale $A$, però in diverse applicazioni è utile che gli elementi campionati siano mantenuti in un ordine casuale.

Vorremmo, più precisamente, che fissato un qualunque sottinsieme $S \subseteq A$ di cardinalità $s$, tutte le possibili permutazioni degli elementi di $S$ siano equiprobabili. Formalmente, indicando con $\vec{S} = \langle x_1, x_2, \ldots, x_s \rangle$ la versione ordinata del sample e con $\mathcal{P}(s)$ l'insieme di tutte le permutazioni di $s$ elementi, richiediamo che:
$$\forall \pi \in \mathcal{P}(s), \qquad \Pr(\vec{S} = \pi (S)\mid\mathcal{S} = S) = \frac{1}{s!}$$

### **Algoritmo di FY_Shuffle**
L'algoritmo di **FY_Shuffle** serve a risolvere la seconda proprietà. Dato un insieme $A$ di $n$ elementi memorizzati in un array $\vec{A} = \vec{A}[1:n]$, l'algoritmo di **FY_Shuffle** produce in modo iterativo una permutazione u.a.r. $\vec{A_\pi}$ di $A$.

<img src="img/FY_shuffle.png" alt="FY_Shuffle" width="300"/>

In pratica l'algoritmo, per generare il nuovo array random, parte copiando l'array originale in $\vec{A_\pi}$. Dopodiché, ad ogni iterazione $i$ sceglie un indice $j$ a caso tra $1$ e $i$ e scambia $\vec{A_\pi}[i]$ con $\vec{A_\pi}[j]$.

Assumendo che il costo di estrarre un numero j u.a.r. tra $1$ e $i$ sia $O(1)$ (anche se abbiamo visto in BD che in realtà dovrebbe essere $O(\log i)$ bit-level), il costo totale dell'algoritmo è $O(n)$.

Si dimostra che l'algoritmo di **FY_Shuffle** produce una permutazione u.a.r. di $A$.

> **Lemma 2.1**  
> 
> Alla fine di ogni round $i = 1, 2, \ldots, n$ dell'algoritmo di **FY_Shuffle**, il prefisso $\vec{A_\pi}[1:i]$ è una permutazione uniforme random dei primi $i$ elementi dell'array iniziale $\vec{A}$.

> **Dimostrazione**  
> 
> La dimostrazione procede per induzione su $i$.
>
> Fissiamo una qualunque permutazione $\langle a_1, \ldots, a_i \rangle$ dei primi $i$ elementi di $\vec{A}$, vogliamo calcolare, alla fine del round $i$, la probabilità dell'evento $$\varepsilon_i = \{\vec{A_\pi}[1:i] = \langle a_1, \ldots, a_i \rangle\}$$
> (ossia la probabilità che data una qualsiasi permutazione dei primi $i$ elementi di $\vec{A}$, alla fine del round $i$ il prefisso $\vec{A_\pi}[1:i]$ sia proprio quella permutazione). Vorremmo ovviamente che tale probabilità fosse $1/i!$.
> 
> - **P.B. $i = 1$**: in tal caso stiamo considerando solo il primo elemento dell'array, che resterà in quella posizione con probabilità 1. Quindi $\Pr(\varepsilon_1) = 1 = \frac{1}{1!}$, pertanto il passo base è verificato.
>
> - **P.I.**: applicando l'ipotesi induttiva fino al passo $i-1$, sappiamo che $$\Pr(\varepsilon_{i-1}) = \Pr(\vec{A_\pi}[1:i-1] = \langle a_1, \ldots, a_{i-1} \rangle) = \frac{1}{(i-1)!}$$  
> Al passo i-esimo scelgo $j$ in modo u.a.r. in $[i]$. Allora ci sono due possibili casi disgiunti ed "esaustivi" da considerare (esaustivi nel senso che soltanto questi due casi possono verificarsi nella generazione del prefisso $\vec{A_\pi}[1:i]$):
> - **Caso 1**: $j = i$: in tal caso non viene effettuato alcuno scambio, quindi la prima parte del prefisso $\vec{A_\pi}[1:i-1]$ resta invariata. Poiché la scelta di $j$ è del tutto indipendente dalla costruzione del prefisso $\vec{A_\pi}[1:i-1]$:
> $$\Pr(\varepsilon_i) = \Pr(\varepsilon_{i-1}) \cdot \Pr(j = i) = \frac{1}{(i-1)!} \cdot \frac{1}{i} = \frac{1}{i!}$$
> - **Caso 2**: $j = k$ con $k < i$: in tal caso viene effettuato uno scambio tra $\vec{A_\pi}[i]$ e $\vec{A_\pi}[k]$. 
> Supponiamo di voler ottenere alla fine del round $i$ $\vec{A}_{\pi}[1:i]=\langle a_1,\ldots,a_i\rangle.$.  
>Affinché ciò accada, prima dello scambio, l'elemento $a_i$ doveva essere in posizione $k$. Ma quindi, prima dello scambio, l'array $\vec{A_\pi}$ doveva essere tale che $\vec{A_\pi}[1:i-1] = \langle a_1, \ldots, a_{k-1}, a_i, a_{k+1}, \ldots, a_{i-1} \rangle$.  
>Ma quindi abbiamo determinato una specifica permutazione del prefisso al passo $i-1$, che per ipotesi induttiva ha probabilità $1/(i-1)!$. Poiché di nuovo la scelta di $j$ è del tutto indipendente dalla costruzione del prefisso $\vec{A_\pi}[1:i-1]$, anche in questo caso:
> $$\Pr(\varepsilon_i) = \Pr(\varepsilon_{i-1}) \cdot \Pr(j = k) = \frac{1}{(i-1)!} \cdot \frac{1}{i} = \frac{1}{i!}$$
> $\blacksquare$

### **Algoritmo di FY_Sample**
Per soddisfare anche la prima proprietà, si definisce l'algoritmo di **FY_Sample**. L'idea dietro l'algoritmo è semplicissima: si esegue anzitutto **FY_Shuffle** per ottenere una permutazione u.a.r. $\vec{A_\pi}$ di $A$, e poi si restituisce come sample i primi $s$ elementi di $\vec{A_\pi}$, ossia $S = \vec{A_\pi}[1:s]$. 

<img src="img/FY_sample.png" alt="FY_Sample" width="300"/>

Chiaramente eseguire soltanto **FY_Shuffle** fino all'elemento $s$ non sarebbe stato sufficiente a garantire la prima proprietà in quanto non avremmo preso in considerazione gli elementi di $\vec{A_\pi}$ a partire da $s+1$ fino a $n$, per questo deve essere eseguito nell'intero array.

> **Lemma 2.2**  
> 
> Sia $\vec{A}[1:n]$ un vettore di n numeri distinti e sia $s \in [n]$. L'algoritmo di **FY_Sample** genera, su input ($\vec{A}$, $s$), un random sample $\mathcal{S}$ la cui distribuzione soddisfa le due seguenti proprietà:
> 1. Inteso come sottoinsieme non ordinato, $\mathcal{S}$ è uniformemente distribuito sullo spazio $\binom{n}{s}$ di tutti i possibili sample di dimensione $s$ (ossia $\forall S \subseteq A : |S| = s, \Pr(\mathcal{S} = S) = 1/\binom{n}{s}$).
> 2. Intendendo $\mathcal{S}$ come vettore $\vec{S}[1:s]$, questo è ordinato in modo u.a.r. rispetto a tutti i possibili $s!$ ordinamenti di $S$ (ossia $\forall \pi \in \mathcal{P}_s, \Pr(\vec{S} = \pi(S)\mid\mathcal{S} = S) = 1/s!$).
>
> **Dimostrazione**  
> 
> **La proprietà 1.** del teorema si dimostra sfruttando la proprietà di **FY_Shuffle** per cui 
> $$
> \forall \pi \in \mathcal{P}_n,
> \qquad
> \Pr(\vec{A}_{\pi} = \pi(A))
> =
> \frac{1}{n!}
> $$
> Fissiamo ora un sottinsieme $S = {x_1, \ldots, x_s}$ di cardinalità $s$, ci chiediamo qual è la probabilità di ottenere esattamente questo sample (in questo caso non stiamo guardando l'ordinamento degli elementi, ma solo il fatto che $S$ sia il sample restituito dall'algoritmo).  
> Casi favorevoli vs Casi possibili: S viene restituito quando i primi s posti della permutazione contengono esattamente gli elementi di S ($x_1, \ldots, x_s$) in qualsiasi posizione (-> $s!$) mentre gli altri $n-s$ elementi di $A$ occupano le rimanenti $n-s$ posizioni sempre in qualsiasi ordine (-> $(n-s)!$).  
> Per quanto riguarda i casi possibili, poiché come detto e dimostrato **FY_Shuffle** restituisce ogni permutazione di $A$ con probabilità $1/n!$, allora i casi possibili sono proprio tutte le permutazioni di $A$ (-> $n!$). Quindi:
> $$
> \Pr(\mathcal{S} = S)
> =
> \frac{s!(n-s)!}{n!}
> =
> \frac{1}{\binom{n}{s}}
> $$
> dimostrando quindi che ogni sample di dimensione $s$ è restituito con la stessa probabilità, ossia che $\mathcal{S}$ è uniformemente distribuito su tutti i possibili sample di dimensione $s$.  
>
> **La proprietà 2.** segue direttamente dalla proprietà di **FY_Shuffle**. Poiché la proprietà di **FY_Shuffle** garantisce che venga generata una permutazione u.a.r. dell'intero array $\vec{A}$, e poiché stiamo considerando i primi $s$ elementi di tale permutazione come parte di $S$ (ossia $S = \vec{A_\pi}[1:s]$), allora è immediato concludere che tutti i loro $s!$ ordinamenti sono equiprobabili.
> 
> $\blacksquare$

## **Reservoir Sampling**
Si considera adesso un modello Streaming, dove l'algoritmo non ha accesso a tutto il dataset $A$ fin dall'inizio, ma riceve gli elementi di $A$ uno alla volta in un flusso continuo, potenzialmente infinito:
$$\text{Streaming input: } a_1, a_2, a_3, \ldots, a_k, \ldots$$
In particolare si ha che al round $k \ge 1$ l'algoritmo riceve l'elemento $a_k$ e deve decidere se includerlo o meno nel sample.

Il problema quindi si trasforma, lo descriviamo formalmente come segue: dato uno Streaming input, si vuole generare e mantenere, $\forall \text{ round } k \ge s$, un sample $\mathcal{S}(s,k)$ di dimensione $s$ che sia uniformemente distribuito su tutti i possibili sample di dimensione $s$ estratti dai primi $k$ elementi dello streaming, ossia:
$$\forall S' \in \binom{k}{s}, \qquad \Pr(\mathcal{S}(s,k) = S') = \frac{1}{\binom{k}{s}}$$
dove $\in \binom{k}{s}$ è un abuso di notazione per indicare che $S'$ è un sottinsieme di cardinalità $s$ dei primi $k$ elementi dello streaming, ossia $S' \subseteq \{a_1, a_2, \ldots, a_k\}$ e $|S'| = s$. Inoltre di nuovo attenzione a non confondere $\mathcal{S}(s,k)$, che è una variabile aleatoria che rappresenta il sample prodotto dall'algoritmo al round $k$, con $S'$, che è un sample fisso, deterministico, di dimensione $s$ estratto dai primi $k$ elementi dello streaming.

L'algoritmo **Reservoir Sampling** soddisfa questa proprietà di uniformità, e lo fa in modo molto semplice: i primi s elementi del flusso vengono inseriti deterministicamente all'interno del sample, mentre per ogni elemento $a_k$ con $k > s$ viene estratto un numero $j$ u.a.r. tra $1$ e $k$. Se $j \le s$, allora $a_k$ viene inserito nel sample al posto dell'elemento che si trova in posizione $j$, altrimenti $a_k$ viene scartato. Chiaramente, all'aumentare dello scorrere del flusso, la probabilità di inserire un nuovo elemento nel sample diminuisce, ma questo come vedremo garantisce che il sample sia sempre uniforme sui primi $k$ elementi dello streaming.

<img src="img/Reservoir_Sampling.png" alt="Reservoir Sampling" width="300"/>

**Complessità**: l'algoritmo richiede di memorizzare solo il sample di dimensione $s$ e l'ultimo elemento dello streaming, quindi la complessità spaziale è $O(s)$ (non richiede di salvare tutto lo streaming, sarebbe eccessivamente costoso). 


> **Teorema 1**  
> 
> $\forall s \ge 1, \forall k \ge s$, $\forall S' \in \binom{k}{s}$, la probabilità che **Reservoir Sampling** generi proprio $S'$ è esattamente $1/\binom{k}{s}$.
>
> **Dimostrazione**  
> 
> La dimostrazione procede per induzione su $k$.
> - **P.B. $k = s$**: in tal caso i primi $s$ elementi del flusso vengono inseriti deterministicamente all'interno di $\mathcal{S}(s,k)$. Esiste un solo sottinsieme possibile di dimensione $s$ estratto dai primi $s$ elementi dello streaming, ossia $S' = \{a_1, a_2, \ldots, a_s\}$, e questo è proprio il sample restituito dall'algoritmo. Quindi $\Pr(\mathcal{S}(s,s) = S') = 1 = 1/\binom{s}{s}$, pertanto il passo base è verificato.
>
> - **P.I.**: assumiamo che l'ipotesi induttiva valga fino al passo $k$, per cui vale che dopo aver processato i primi $k$ elementi dello streaming, allora ogni sottinsieme di dimensione $s$ estratto dai primi $k$ elementi dello streaming ha probabilità $1/\binom{k}{s}$ di essere generato come output dell'algoritmo $\mathcal{S}(s,k)$.  
> Consideriamo quindi l'elemento $a_{k+1}$ che arriva al round $k+1$. Allora l'algoritmo genera un indice $j$ u.a.r. tra $1$ e $k+1$. Ci sono due casi disgiunti ed "esaustivi" da considerare per determinare il nuovo sample $\mathcal{S}(s,k+1)$:
> - **Caso 1**: $a_{k+1} \notin S'$: stiamo dicendo che l'elemento del flusso non viene inserito nel generico sottinsieme deterministico $S'$. Ora, affinché $\mathcal{S}(s,k+1) = S'$, è quindi necessario che $S'$ sia già stato generato al passo $k$, e ciò è vero se e solo se si ha la congiunzione dei due eventi:  
>   1. **l'elemento $a_{k+1}$ non viene inserito nel sample**: questo accade quando $j > s$, con probabilità $(k+1-s)/(k+1)$.
>   2. **il sample generato al passo $k$  è proprio $S'$**: questo accade con probabilità $Pr(\mathcal{S}(s,k) = S') = 1/\binom{k}{s}$ per ipotesi induttiva.
>   
>    Moltiplicando le probabilità dei due eventi otteniamo:
>
>     $$
>     \begin{aligned}
>     Pr(\mathcal{S}(s,k+1)=S')
>     &= Pr(\mathcal{S}(s,k)=S') \cdot Pr(j>s) \\
>     &= \frac{1}{\binom{k}{s}} \cdot \frac{k+1-s}{k+1} \\
>     &= \frac{1}{\binom{k+1}{s}}
>     \end{aligned}
>     $$
>
>     dove l'ultimo passaggio deriva dal fatto che
>     $(k+1-s)! = (k-s)!(k+1-s)$
>     e
>     $(k+1)! = k!(k+1)$.
>
> - **Caso 2**: $a_{k+1} \in S'$: cioè il nuovo elemento deve comparire nel sample finale (generico, deterministico). Ora, per ottenere un sample $S'$ che contiene $a_{k+1}$ e far sì che $S' = \mathcal{S}(s,k+1)$, è necessario che il sample precedente $\mathcal{S}(s,k)$ contenesse gli altri $s-1$ elementi di $S'$, più un elemento extra $X$ che sarà sostituito da $a_{k+1}$ al round $k+1$.  
> Si definisce $S_{fixed} = S' \setminus \{a_{k+1}\}$, ossia il sottoinsieme di $S'$ che contiene tutti gli $s-1$ elementi (fissi) di $S'$ tranne $a_{k+1}$.  
> Ma allora il vecchio sample $\mathcal{S}(s,k)$ doveva essere proprio $S_{fixed} \cup \{X\}$, con $X$ che è un elemento qualsiasi del flusso $a_1, a_2, \ldots, a_k$, ma che chiaramente non può appartenere a $S_{fixed}$ in quanto gli elementi in $S_{fixed}$ non cambieranno al round $k+1$ --> le possibili scelte per $X$ sono quindi $k - (s-1) = k+1-s$.  
> Quindi esistono esattamente $k+1-s$ sample possibili $\mathcal{S}(s,k)$ che possono portare a $S'$ al round $k+1$ tramite la sostituzione di $X$ con $a_{k+1}$. Poiché ognuna di queste configurazioni è mutualmente esclusiva (o una o l'altra o l'altra ancora etc..), la probabilità totale di ottenere $S'$ al round $k+1$ è data dalla somma delle probabilità di ognuna di queste configurazioni, si ottiene quindi:
>     $$
>     \Pr(\mathcal{S}(s,k+1) = S')
>     =
>     \sum_{x \in \{a_1, \ldots, a_k\}}
>     \Pr(\mathcal{S}(s,k)=S_{fixed}\cup\{x\})
>     \cdot
>     \Pr(a_{k+1}\text{ sostituisce }x)
>     $$
>   (dove si ha x piccolo nella sommatoria perché stiamo "fissando" il valore della v.a. X).  
> Per quel che riguarda la prima probabilità, per ipotesi induttiva, ogni sample di dimensione  $s$ estratto dai primi $k$ elementi dello streaming ha probabilità $1/\binom{k}{s}$ di essere generato come output dell'algoritmo $\mathcal{S}(s,k)$, quindi anche $\Pr(\mathcal{S}(s,k)=S_{fixed}\cup\{x\}) = 1/\binom{k}{s}$.  
> La seconda probabilità ci dice che anzitutto $a_{k+1}$ deve essere inserito nel sample, e questo accade quando $j \le s$, con probabilità $s/(k+1)$. Inoltre deve avvenire che $a_{k+1}$ sostituisca proprio $x$, e questo accade quando $j$ è esattamente uguale alla posizione di $x$ all'interno del sample, con probabilità $1/s$. Quindi, trattandosi di due eventi indipendenti, la seconda probabilità è data da $\Pr(a_{k+1}\text{ sostituisce }x) = \Pr(j \le s) \cdot \Pr(j\text{ è la posizione di }x) = \frac{s}{k+1} \cdot \frac{1}{s} = \frac{1}{k+1}$.  
> Sostituendo le due probabilità nella sommatoria otteniamo:
>   $$ \begin{aligned} \Pr(\mathcal{S}(s,k+1) = S') &= (k-s+1) \cdot \frac{1}{\binom{k}{s}} \cdot \frac{1}{k+1} \\ &= (k-s+1) \cdot \frac{s!(k-s)!}{k!} \cdot \frac{1}{k+1} \\ &= \frac{1}{\binom{k+1}{s}} \end{aligned} $$
> 
> $\blacksquare$

### **Permuting Reservoir Sampling Algorithm**
Il nuovo obiettivo adesso è definire una variante di Reservoir Sampling che, oltre a garantire la proprietà di uniformità sui sample, garantisca anche la proprietà di uniformità sull'ordinamento degli elementi all'interno del sample. 

Formalmente, si vuole un algoritmo che risolva il problema per cui, dato uno Streaming input $a_1, \cdots, a_k, \cdots$, si vuole generare e mantenere, $\forall \text{ round } k \ge 1$, un sample ordinato $\vec{\mathcal{S}}(s,k)$ di dimensione $s$ tale che:
1. Inteso come sottoinsieme non ordinato, $\mathcal{S}(s,k)$ è uniformemente distribuito sullo spazio $\binom{k}{s}$ di tutti i possibili sample di dimensione $s$ estratti dai primi $k$ elementi dello streaming  
   (ossia $\forall S' \in \binom{k}{s}, \qquad \Pr(\mathcal{S}(s,k) = S') = 1/\binom{k}{s}$).
2. Intendendo $\vec{\mathcal{S}}(s,k)$ come vettore ordinato, condizionatamente al fatto che il sample non ordinato sia un certo insieme $S'$, allora l'ordine degli elementi in $\vec{\mathcal{S}}(s,k)$ è uniforme tra tutte le $s!$ permutazioni di $S'$    
   (ossia $\forall \pi \in \mathcal{P}_s, \forall S' \in \binom{k}{s}, \qquad \Pr(\vec{\mathcal{S}}(s,k) = \pi(S')\mid\mathcal{S}(s,k) = S') = 1/s!$).

L'algoritmo è chiamato **PRSA Permuting Reservoir Sampling Algorithm** e combina RSA con Fisher-Yates: l'algoritmo segue due fasi:
1. **Riempimento del Reservoir**: per i primi $s$ elementi dello streaming, ognuno di essi viene inserito in fondo al sample per poi essere scambiato con un elemento a caso $i \in_{u} [k]$, sfruttando quindi la logia di **FY_Shuffle** per garantire che i primi $s$ elementi del sample siano ordinati in modo u.a.r. rispetto a tutti i possibili $s!$ ordinamenti. 
2. **Aggiornamento del Reservoir**: all'arrivo di ogni elemento nuovo, si sceglie $j \in_{u} [k]$. Se $j \gt s$ --> l'elemento viene scartato. Se $j \le s$ --> viene rimosso $a_j$ dal sample, si fanno scorrere a sinistra tutti gli elementi che si trovano a destra di $a_j$ fino ad $a_{s}$, dopodiché si inserisce proprio in posizione $s$ (rimasta vuota) il nuovo elemento $a_k$, per poi applicare la logica di **FY_Shuffle** scegliendo $h \in_{u} [s]$ e scambiando $a_k$ con l'elemento in posizione $h$.

<img src="img/PRSA.png" alt="PRSA" width="350"/>

L'intuizione dietro l'integrazione con FY_Shuffle è che questo ha la seguente proprietà: se i primi $s-1$ elementi sono già una permutazione uniforme, allora inserire un nuovo elemento in una posizione casuale produce una nuova permutazione uniforme di $s$ elementi.

**Complessità**: a livello spaziale esattamente come Reservoir Sampling: $O(s)$. A livello temporale invece ogni round richiede una scelta random, in caso $j \le s$ è necessario effettuare uno scorrimento di tutti gli elementi a destra di $a_j$ (nel caso peggiore $s-1$ elementi) più un eventuale scambio, quindi la complessità temporale per round è $O(s)$.

> **Teorema 2**
>
> PRSA mantiene, ad ogni round $k \ge s$, un vettore $\vec{\mathcal{S}}(s,k)$ di dimensione $s$ tale che:
> 1. intendendo $\mathcal{S}(s,k)$ come sottoinsieme non ordinato, questo è scelto uniformemente tra tutti i sottinsiemi di size $s$ estratti dai primi $k$ elementi dello streaming (ossia $\forall S' \in \binom{k}{s}, \Pr(\mathcal{S}(s,k) = S') = 1/\binom{k}{s}$);
> 2. intendendo $\vec{\mathcal{S}}(s,k)$ come vettore ordinato, condizionatamente al fatto che il sample non ordinato sia un certo insieme $S'$, allora l'ordine degli elementi in $\vec{\mathcal{S}}(s,k)$ è uniforme tra tutte le $s!$ permutazioni di $S'$  
> (ossia $\forall \pi \in \mathcal{P}_s, \forall S' \in \binom{k}{s}, \Pr(\vec{\mathcal{S}}(s,k) = \pi(S')\mid\mathcal{S}(s,k) = S') = 1/s!$).
>
> **Dimostrazione**
>
> La prima proprietà segue direttamente dal Reservoir Sampling classico, infatti PRSA quando considera se accettare o scartare il nuovo elemento $a_k$ usa esattamente la stessa regola di RSA (scelgo j u.a.r. tra $1$ e $k$, se $j \le s$ accetto, altrimenti scarto. Non mi interessa la posizione dove lo metto perché RSA lavorava su insiemi, mi basta sapere se lo prendo o meno!), quindi la prima proprietà è soddisfatta poiché vale il **Teorema 1**.
>
> Dimostriamo ora la seconda parte. Si dimostra per induzione su $k$ che, ad ogni round, il vettore $\vec{\mathcal{S}}(s,k)$ è una permutazione uniforme degli elementi presenti nel reservoir $\mathcal{S}(s,k)$.
> 1. **P.B. $k = s$**: quando arrivano i primi $s$ elementi dello streaming, PRSA li inserisce uno alla volta, ma dopo ogni inserimento sceglie una posizione casuale e fa uno swap. Per il **Lemma 2.1** (Lemma di FY_Shuffle), dopo aver inserito i primi $s$ elementi, il vettore $\vec{\mathcal{S}}(s,s)$ è una permutazione uniforme dei primi $s$ elementi dello streaming, quindi il passo base è verificato.
> 2. **P.I.**: supponiamo che al round $k-1$ il reservoir sia ordinato uniformemente, ossia condizionatamente al suo contenuto ogni ordinamento possibile degli $s$ elementi ha probabilità $1/s!$ di essere proprio quello del reservoir. A questo punto arriva un nuovo elemento $a_k$, si hanno due casi disgiunti ed esaustivi:
> - **Caso 1**: $a_k$ viene scartato --> PRSA non modifica il reservoir: $\vec{\mathcal{S}}(s,k) = \vec{\mathcal{S}}(s,k-1)$, quindi l'ordine resta quello del round precedente, che per ipotesi induttiva era già uniforme.
> - **Caso 2**: $a_k$ entra nel reservoir, questo accade quando $j \le s$. In questo caso viene rimosso un elemento $a_j$ dal reservoir, si fanno scorrere a sinistra tutti gli elementi a destra di $a_j$ fino ad $a_{s}$, dopodiché si inserisce proprio in posizione $s$ (rimasta vuota) il nuovo elemento $a_k$, per poi scegliere $h \in_{u} [s]$ e scambiare $a_k$ con l'elemento in posizione $h$.  
> Ora un'intuizione importante: **gli $s-1$ elementi sopravvissuti restano ordinati uniformemente**. Infatti prima dell'arrivo di $a_k$, per ipotesi induttiva, il reservoir era ordinato uniformemente tra tutte le permutazioni degli $s$ elementi. Quando viene rimosso un elemento, gli altri $s-1$ elementi mantengono l'ordine relativo che avevano prima e quindi restano ordinati uniformemente tra loro. Sarebbe a dire che se rimangono gli elementi $x_1, \cdots, x_{s-1}$, allora ogni possibile ordinamento di questi $s-1$ elementi ha probabilità $1/(s-1)!$ di essere quello che si ottiene dopo la rimozione.  
> A questo punto abbiamo quindi $s-1$ elementi sopravvissuti ordinati uniformemente, il nuovo elemento $a_k$ da inserire e una posizione causale $h \in_{u} [s]$ da scegliere per lo swap. L'algoritmo mette $a_k$ in posizione $s$ e poi lo scambia con l'elemento in posizione $h$, ma questo equivale a inserire $a_k$ in una posizione casuale tra le $s$ possibili.  
> Contiamo quindi le probabilità. Fissiamo una qualunque permutazione finale desiderata $\vec{T} = \langle t_1, t_2, \ldots, t_s \rangle$ degli s elementi finali. Supponiamo che $a_k$ compaia in posizione $r$ dentro $\vec{T}$, con $r \in [s]$. Per ottenere $\vec{T}$ alla fine del round $k$, devono verificarsi due eventi indipendenti:
>     1. gli $s-1$ elementi diversi da $a_k$ devono essere già ordinati nell'unico ordine compatibile con $\vec{T}$, e questo accade con probabilità $1/(s-1)!$ per l'intuizione spiegata sopra;
>     2. $a_k$ deve essere inserito proprio in posizione $r$, e questo accade quando $h = r$, con probabilità $1/s$. 
> 
>    Moltiplicando le due probabilità otteniamo:
> $$\Pr(\vec{\mathcal{S}}(s,k) = \vec{T}) = \frac{1}{(s-1)!} \cdot \frac{1}{s} = \frac{1}{s!}$$
> $\blacksquare$

## Distributed Sampling: Merging Protocol
Si torna ora al mondo distribuito.

Si considera ora una **Star Network**: si ha un **Server Centrale** $C$ e $h \ge 2$ **Nodi Periferici** $P_1, P_2, \ldots, P_h$ che comunicano solo con il server $C$. Si ha quindi un communication graph $G = (V, E)$ dove:
$$ V = \{C, P_1, P_2, \ldots, P_h\}, \qquad E = \{(C, P_i) : i \in [h]\}$$

Si vuole risolvere il seguente problema: $C$ deve mantenere un sample $\mathcal{S}$ di un datased globale $D$, che è l'unione disgiunta (è partizionato) di $h$ sottodataset $D_1, D_2, \ldots, D_h$ memorizzati rispettivamente nei nodi periferici $P_1, P_2, \ldots, P_h$. $C$ non ha alcun accesso diretto ai dati, ma può comunicare con i nodi periferici per ottenere informazioni su di essi. 

**Soluzione Naive**: ogni nodo periferico $P_i$ invia $D_i$ al server $C$, che poi esegue un algoritmo di sampling su $D$ per ottenere $\mathcal{S}$. Il problema di questo approccio non è solo il fatto che $D$ è molto grande e quindi difficile da mantenere in memoria, ma soprattutto il fatto che la **communication complexity** è O($|D|$).

**Soluzione migliore**: ogni nodo $P_i$ genera un sample locale $\mathcal{S}_i$ di dimensione $s$ estratto da $D_i$ con uno degli algoritmi visti in precedenza, ed invia al server $C$ solo il sample $\mathcal{S}_i$ invece di tutto il dataset $D_i$. A partire da questi $h$ sample locali, il server $C$ genera un sample globale $\mathcal{S}$ di dimensione $s$ che mantenga le proprietà di uniformità sui sample.

La complessità del protocollo visto sopra è $O(\sum_{i=1}^h s) = O(hs)$, che è molto più efficiente rispetto alla soluzione naive, soprattutto quando $D$ è molto grande.

**Soluzione ancora migliore**: noi mostreremo un protocollo ancora più efficiente con la seguente idea: ogni nodo periferico $P_i$ genera un sample locale $\mathcal{S}_i$ e $C$, secondo una logica locale, decide il numero $s_i$ di elementi di $\mathcal{S}_i$ da prendere dal nodo $P_i$ per formare il sample globale $\mathcal{S}$. Con questo approccio vedremo che è possibile ottenere un protocollo con complessità $O(h+s)$, che è ottimale in quanto è necessario almeno $O(h)$ per comunicare con tutti i nodi periferici e $O(s)$ per restituire un sample di dimensione $s$.

Ha senso l'idea di scegliere il numero di elementi da prendere da ogni sample locale? Sì, perché se ad esempio se abbiamo solo due nodi periferici $P_1$ e $P_2$ con rispettivamente $|D_1| = 100$ e $|D_2| = 900$, e vogliamo un sample globale di dimensione $s = 10$, allora è chiaro che ha senso prendere più elementi da $P_2$ rispetto a $P_1$ per cercare di avere un campione rappresentativo del dataset originale.

### Ipergeometrica: perché ci serve
Fissiamoci nel caso in cui si hanno solo due nodi periferici $P_1$ e $P_2$ con rispettivamente $|D_1| = n_1$ e $|D_2| = n_2$, con $n_1 + n_2 = n = |D|$. Supponiamo di voler ottenere un sample globale di dimensione $s$.

**Problema: non possiamo fissare arbitrariamente le quantità di elementi da prendere da ogni nodo periferico, altrimenti rischiamo di non avere un campione rappresentativo del dataset globale.** Nell'esempio di prima un sample uniforme tenderà infatti ad avere circa 9 elementi da $P_2$ e 1 elemento da $P_1$, ma non è sempre esattamente così, a volte potrebbe contenerne 8 e 2, altre 10 e 0 etc...

Per questo è necessario fare affidamento a una distribuzione di probabilità che modelli correttamente la situazione e permetta di ottenere un sample che sia poi uniforme su tutti i possibili sample di dimensione $s$ estratti da $D$.

La distribuzione corretta del numero di elementi da prendere da un nodo periferico è la **distribuzione ipergeometrica**:
$$K \sim \text{Hypergeometric}(n, n_1, s)$$
dove: 
- $n = n_1 + n_2$ è la dimensione totale del dataset
- $n_1 = |D_1|$ è la dimensione del sottodataset del nodo periferico $P_1$
- $s$ è la dimensione del sample globale che vogliamo ottenere
- $K$ è la variabile aleatoria che rappresenta il numero di elementi da prendere da $P_i$ per formare il sample globale $\mathcal{S}$.

La formula è la seguente:
$$\Pr(K = k) = \frac{\binom{n_1}{k} \binom{n_2}{s-k}}{\binom{n}{s}}$$
dove $k \in [\max(0, s-n_2), \min(s, n_1)]$ (ossia non posso prendere più elementi di quanti ne esistono in $D_2$ --> dato che da $D_2 prendo $s-k$ elementi deve valere $s-k \leq n_2$, portando $k$ dall'altra parte si ottiene $k \geq \max(0, s-n_2)$, e non posso prendere da $D_1$ più elementi di quanti ne esistono in $D_1$ --> $k \leq \min(s, n_1)$).

Questa formula ci dice: tra tutti i possibili sample globali di size $s$, qual è la probabilità che esattamente $k$ elementi provengano da $P_1$ mentre gli altri $s-k$ provengono da $P_2$? 

L'ipergeometrica è la distribuzione corretta da utilizzare in questo contesto perché modella esattamente il processo di estrazione di un campione senza reinserimento da una popolazione finita che è divisa in due classi. L'uso della binomiale invece non sarebbe corretto perché la binomiale modella un processo di estrazione con reinserimento (vedi in img binom1, 2 e 3 per l'esempio del perché binomiale non andava bene).

Si noti come la semplificazione del problema a solo due nodi periferici è stata **necessaria** per poter utilizzare la distribuzione ipergeometrica, che è definita proprio per il caso di due classi. Se avessimo più di due nodi periferici, allora dovremmo utilizzare una generalizzazione della distribuzione ipergeometrica, chiamata **multinomial hypergeometric distribution**, la logica è la stessa ma i calcoli sono più complessi.

Vedremo che, nell'algoritmo, l'idea è che il server $C$ generi un numero casuale $K$ secondo la probabilità data dall'ipergeometrica, per cui quindi il numero di elementi da prendere da $P_1$ seguirà tale distribuzione.

Prima di vedere l'algoritmo, vediamo alcune proprietà importanti della distribuzione ipergeometrica che ci torneranno utili nella dimostrazione di correttezza dell'algoritmo:

<img src="img/ip1.png" alt="Ipergeometrica" width="400"/>

Sia $X \sim \text{Hypergeometric}(n, m, s)$. Sia $Y_1, \cdots, Y_s$ sono $s$ variabili aleatorie bernoulliane (dove $Y_i = 1$ se l'elemento $i$ del sample globale proviene da $P_1$ (estrazione con successo), altrimenti $Y_i = 0$ se proviene da $P_2$), allora si ha che:
1. $\Pr(Y_1 = 1) = \frac{m}{n}$, ossia la probabilità che un elemento del sample globale provenga da $P_1$ è pari alla frazione di elementi di $P_1$ rispetto al totale.
2. $\Pr(Y_2 = 1 \mid Y_1 = 1) = \frac{m-1}{n-1}$
3. $\Pr(Y_2 = 1 \mid Y_1 = 0) = \frac{m}{n-1}$

**Exchangeability**: l'ordine specifico degli 0 o degli 1 delle bernoulliane nella costruzione del campione non cambia la probabilità di ottenere quel campione, conta solo quanti 1 ci sono! 

Più formalmente, si dice che la sequenza di variabili aleatorie $Y_1, \cdots, Y_s$ è **exchangeable**, ossia la probabilità di una qualsiasi sequenza dipende solo dal numero $k$ di successi (1) e non dall'ordine in cui si verificano:
$$\Pr(Y_1 = y_1, \cdots, Y_s = y_s) = \frac{m! (n-m)! (n-s)!}{n! (m-k)! (n-m-s+k)!}$$
dove la formula deriva dal fatto che stiamo facendo $k$ estrazioni della classe di successo con $m$ elementi, $s-k$ estrazioni della classe di insuccesso con $n-m$ elementi, e poi stiamo scegliendo $s$ elementi da un totale di $n$.  

<p align="center">
    <img src="img/exc1.png" alt="Exchangeability" width="45%"/>
    <img src="img/exc2.png" alt="Exchangeability" width="50%"/>
</p>

(l'uso del fattoriale decrescente deriva dal fatto che stiamo facendo estrazioni senza reinserimento, quindi ogni volta che estraiamo un elemento, il numero di elementi rimanenti nella classe di successo o insuccesso diminuisce di 1 -> non posso usare la binomiale, non sto prendendo tutti i possibili sottinsiemi di dimensione $k$ da $m$).

### **Weighted Hypergeometric Merging Protocol WHS**
Sia $D = D_1 \cup D_2$ con $|D_1| = n_1$ in $P_1$ e $|D_2| = n_2$ in $P_2$. Sia $s$ la dimensione del sample globale che vogliamo ottenere. Il protocollo WHS funziona come segue:
1. $C$ richiede $n_1 = |D_1|$ e $n_2 = |D_2|$ a $P_1$ e $P_2$ rispettivamente, e calcola $n = n_1 + n_2$.
2. $C$ genera un numero casuale $k$ secondo la distribuzione ipergeometrica $\text{Hypergeometric}(n, n_1, s)$, che rappresenta il numero di elementi da prendere da $P_1$.
3. $C$ invia a $P_1$ il valore $k$ e a $P_2$ il valore $s-k$.
4. $P_i$ genera un sample locale $\mathcal{S}_i$ di dimensione $s$ estratto da $D_i$ con uno degli algoritmi visti in precedenza
5. $P_1$ estrae u.a.r. un sottinsieme $\mathcal{R}_1$ dal suo sample locale $\mathcal{S}_1$ di dimensione $k$ e lo invia a $C$. $P_2$ estrae u.a.r. un sottinsieme $\mathcal{R}_2$ dal suo sample locale $\mathcal{S}_2$ di dimensione $s-k$ e lo invia a $C$.
6. $C$ restituisce il sample globale $\mathcal{S} = \mathcal{R}_1 \cup \mathcal{R}_2$.

> **Main Theorem**
>
> Dato un dataset globale $D = D_1 \cup D_2$, distribuito tra i due nodi $P_1$ e $P_2$, il protocollo WHS genera un sample globale $\mathcal{S}$ di dimensione $s$ che è uniformemente distribuito su tutti i possibili sample di dimensione $s$ estratti da $D$, ossia:
> $$\forall X \subseteq D, |X| = s: \quad \Pr( \mathcal{S} = X ) = \frac{1}{\binom{n}{s}}$$
> dove $n = |D| = n_1 + n_2$.
>
>
> **Dimostrazione**
>
> Per dimostrare la correttezza del protocollo, fissiamo un qualunque sottinsieme $X \subseteq D$ di dimensione $s$, e mostriamo che $\Pr(\mathcal{S} = X) = \frac{1}{\binom{n}{s}}$ (la probabilità che WHS generi proprio $X$ è uniforme su tutti i possibili sample di dimensione $s$).
>
> Ora, $X$ contiene alcuni elementi da $D_1$ e alcuni elementi da $D_2$. Supponiamo che $k = |X \cap D_1|$ sia il numero di elementi di $X$ che provengono da $D_1$, e quindi $s-k = |X \cap D_2|$ sia il numero di elementi di $X$ che provengono da $D_2$.  
> Chiamiamo $X_1 = X \cap D_1$ e $X_2 = X \cap D_2$, quindi $X = X_1 \cup X_2$ con $|X_1| = k$ e $|X_2| = s-k$. 
>
> Affinché il protocollo generi proprio $X$ come sample globale, devono verificarsi i seguenti tre eventi:
> 1. **Evento $E_1$: il server sceglie proprio $K = k$**: ossia nella generazione del valore dalla distribuzione ipergeometrica, $C$ deve aver deciso di prendere esattamente $k$ elementi da $D_1$. Questo accade con probabilità $\Pr(K = k) = \frac{\binom{n_1}{k} \binom{n_2}{s-k}}{\binom{n}{s}}$ per la definizione di distribuzione ipergeometrica.
> 2. **Evento $E_2$: a partire da P_1 arrivano proprio gli elementi $X_1$ a $C$**: il nodo $P_1$ segue infatti due passaggi; anzitutto estrae un sample locale $\mathcal{S}_1$ di dimensione $s$ da $D_1$, e poi estrae u.a.r. un sottinsieme $\mathcal{R}_1$ di dimensione $k$ da $\mathcal{S}_1$. Per ottenere $X$ è necessario che $X_1 = \mathcal{R}_1$. Questa probabilità si calcola in due pezzi:
>     - anzitutto $\mathcal{S}_1$ deve contenere $X_1$. Poiché $\mathcal{S}_1$ ha size $s$, se contiene già i $k$ elementi fissati di $X_1$, gli altri $s-k$ elementi possono essere scelti tra i restanti $n_1-k$ elementi di $D_1$. Pertanto:
>       $$\Pr(X_1 \subseteq \mathcal{S}_1) = \frac{\binom{n_1-k}{s-k}}{\binom{n_1}{s}}$$
>      - una volta che $X_1$ è contenuto in $\mathcal{S}_1$, allora il nodo deve    selezionare proprio quei $k$ elementi tra gli $s$ presenti in $\mathcal{S}_1$. Poiché questo sottinsieme di dimensione $k$ è scelto in modo u.a.r., allora la probabilità di selezionare proprio $X_1$ è data da $\Pr(\mathcal{R}_1 = X_1 \mid X_1 \subseteq \mathcal{S}_1) = 1/\binom{s}{k}$. Ma quindi:
>       $$\Pr(E_2) = \frac{\binom{n_1-k}{s-k}}{\binom{n_1}{s}} \cdot \frac{1}{\binom{s}{k}}$$
> 3. **Evento $E_3$: a partire da P_2 arrivano proprio gli elementi $X_2$ a $C$**: ragionamento del tutto analogo ad $E_2$. $P_2$ estrae un sample locale $\mathcal{S}_2$ di dimensione $s$ da $D_2$, e poi estrae u.a.r. un sottinsieme $\mathcal{R}_2$ di dimensione $s-k$ da $\mathcal{S}_2$. Vogliamo che $\mathcal{R}_2 = X_2$, dove $|X_2| = s-k$. 
>       - Prima di tutto $\mathcal{S}_2$ deve contenere $X_2$. Poiché $\mathcal{S}_2$ ha size $s$ e $X_2$ ha size $s-k$, per completare $\mathcal{S}_2$ è necessario scegliere altri $s-(s-k) = k$ elementi tra i restanti $n_2-(s-k)$ elementi di $D_2$. Pertanto:
>       $$\Pr(X_2 \subseteq \mathcal{S}_2) = \frac{\binom{n_2-s+k}{k}}{\binom{n_2}{s}}$$
>       - una volta che $X_2$ è contenuto in $\mathcal{S}_2$, allora $P_2$ deve selezionare proprio quei $s-k$ elementi tra gli $s$ presenti in $\mathcal{S}_2$. Poiché questo sottinsieme di dimensione $s-k$ è scelto in modo u.a.r., allora la probabilità di selezionare proprio $X_2$ è data da $\Pr(\mathcal{R}_2 = X_2 \mid X_2 \subseteq \mathcal{S}_2) = 1/\binom{s}{s-k}$. Ma quindi:
>           $$\Pr(E_3) = \frac{\binom{n_2-s+k}{k}}{\binom{n_2}{s}} \cdot \frac{1}{\binom{s}{s-k}}$$
>           questa espressione può essere semplificata sfruttando la seguente identità:
>           $$\binom{N}{K} \binom{K}{r} = \binom{N}{r} \binom{N-r}{K-r}$$
>           applicando questa identità con $N = n_2$, $K = s$ e $r = s-k$, otteniamo:
>           $$\binom{n_2}{s} \binom{s}{s-k} = \binom{n_2}{s-k} \binom{n_2-s+k}{s-(s-k)} = \binom{n_2}{s-k} \binom{n_2-s+k}{k}$$
>           da cui si ottiene: 
>    $$\Pr(E_3) = \frac{\binom{n_2-s+k}{k}}{\binom{n_2}{s}} \cdot \frac{1}{\binom{s}{s-k}} = \frac{\binom{n_2-s+k}{k}}{\binom{n_2}{s-k} \binom{n_2-s+k}{k}} = \frac{1}{\binom{n_2}{s-k}}$$
>
> A questo punto moltiplichiamo le probabilità dei tre eventi, che sono indipendenti, per ottenere la probabilità totale di ottenere $X$ come sample globale:
> $$ \Pr(\mathcal{S} = X) = \Pr(E_1) \cdot \Pr(E_2) \cdot \Pr(E_3)$$
> sostituendo:
> $$ \Pr(\mathcal{S} = X) = \left[\frac{\binom{n_1}{k} \binom{n_2}{s-k}}{\binom{n}{s}}\right] \cdot \left[\frac{\binom{n_1-k}{s-k}}{\binom{n_1}{s} \binom{s}{k}}\right] \cdot \left[\frac{1}{\binom{n_2}{s-k}}\right]$$
> Il termine $\binom{n_2}{s-k}$ si semplifica, e otteniamo:
> $$ \Pr(\mathcal{S} = X) = \frac{1}{\binom{n}{s}} \cdot \frac{\binom{n_1}{k} \binom{n_1-k}{s-k}}{\binom{n_1}{s} \binom{s}{k}}$$
> Sfruttiamo ora la seguente identità combinatoria:
> $$ \binom{n_1}{k} \binom{n_1-k}{s-k} = \binom{n_1}{s} \binom{s}{k}$$
> (intuitivamente è vera perché stiamo contando la stessa scelta in due ordini diversi. es. immaginiamo di avere $n_1$ studenti, vogliamo formare un gruppo di $s$ studenti e da quel gruppo scegliere $k$ studenti "speciali". Al lato sinistro prima scelgo i $k$ speciali, poi gli altri $s-k$ studenti normali tra quelli rimasti. Al lato destro invece prima scelgo i $s$ studenti del gruppo, poi scelgo i $k$ speciali tra quelli del gruppo. In entrambi i casi sto contando lo stesso numero di modi per formare il gruppo e scegliere i $k$ speciali, quindi le due espressioni devono essere uguali).
> 
> Applicando questa identità otteniamo:
> $$ \Pr(\mathcal{S} = X) = \frac{1}{\binom{n}{s}} \cdot \frac{\binom{n_1}{s} \binom{s}{k}}{\binom{n_1}{s} \binom{s}{k}} = \frac{1}{\binom{n}{s}}$$
> In conclusione, dal momento che abbiamo fissato qualunque sample finale possibile $X \subseteq D$ di dimensione $s$, e abbiamo dimostrato il protocollo lo produce con probabilità $1/\binom{n}{s}$, allora il protocollo genera un sample globale che è uniformemente distribuito su tutti i possibili sample di dimensione $s$ estratti da $D$.
> 
> $\blacksquare$